# Training LogBERT + DeepSVDD — Google Colab

Melatih model deteksi anomali BGL di Colab (GPU T4 gratis), lalu unduh artefak
untuk dipakai `src/model/predict.py` di project lokal.

**Sebelum mulai:** menu `Runtime` → `Change runtime type` → **GPU (T4)**.

Artefak yang dihasilkan (kompatibel dengan `predict.py`):
`models/logbert/`, `models/svdd.pt`, `models/drain3_state.bin`.

Alur: upload `BGL.log` ke Google Drive → mount → Tahap 1–6 → unduh `models.zip`.

## 0. Cek GPU + install dependency
Colab sudah punya torch & transformers. Hanya `drain3` yang perlu diinstall.

In [ ]:
!nvidia-smi -L
!pip install -q drain3
import torch
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 1. Mount Google Drive & konfigurasi

Upload `BGL.log` (708 MB) ke Google Drive lebih dulu — mis. ke `MyDrive/BGL.log`.
(Upload via Drive jauh lebih andal daripada `files.upload()` untuk file besar.)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# ── Path ────────────────────────────────────────────────────────
RAW_LOG = Path('/content/drive/MyDrive/BGL.log')   # <-- sesuaikan lokasi di Drive
OUT     = Path('/content/models'); OUT.mkdir(exist_ok=True)
LOGBERT_DIR = OUT / 'logbert'
SVDD_MODEL  = OUT / 'svdd.pt'
DRAIN_STATE = OUT / 'drain3_state.bin'

# ── Parameter (HARUS sama dengan config.py lokal) ───────────────
BASE_MODEL='bert-base-uncased'; LOGKEY_PREFIX='LOGKEY_'   # BERT penuh
WINDOW_MINUTES=5; MAX_LOG_KEYS=128; MAX_LENGTH=256
BATCH_SIZE=16; GRAD_ACCUM=1; EPOCHS=20; MLM_PROBABILITY=0.15
TRAIN_RATIO=0.7; SEED=42; SVDD_EPOCHS=100

# ── Mode ────────────────────────────────────────────────────────
LIMIT=None    # None=seluruh dataset; angka kecil utk uji cepat
SMOKE=False   # True=training singkat utk cek program

assert RAW_LOG.exists(), f'{RAW_LOG} tidak ada di Drive. Upload BGL.log dulu.'
print('OK:', RAW_LOG, f'({RAW_LOG.stat().st_size/1e6:.0f} MB)')

## 2. Tahap 1 — Muat BGL.log

In [ ]:
import pandas as pd

rows = []
with open(RAW_LOG, encoding='utf-8', errors='ignore') as f:
    for i, line in enumerate(f):
        if LIMIT and i >= LIMIT: break
        p = line.strip().split(None, 8)
        if len(p) < 9: continue
        rows.append({'is_anomaly': p[0] != '-', 'timestamp_unix': int(p[1]), 'content': p[8]})
df = pd.DataFrame(rows)
df['datetime'] = pd.to_datetime(df['timestamp_unix'], unit='s')
print(f'{len(df):,} baris | anomali {df["is_anomaly"].mean():.2%}')

## 3. Tahap 2 — Drain3 → Log Key
State parser disimpan (WAJIB dipakai ulang saat inference lokal).

In [ ]:
from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig
from drain3.file_persistence import FilePersistence

cfg = TemplateMinerConfig(); cfg.drain_sim_th = 0.4; cfg.drain_depth = 4
miner = TemplateMiner(persistence_handler=FilePersistence(str(DRAIN_STATE)), config=cfg)

keys = []
for i, c in enumerate(df['content']):
    keys.append(miner.add_log_message(c)['cluster_id'])
    if (i+1) % 500_000 == 0: print(f'  {i+1:,} ...')
df['log_key'] = keys
miner.save_state('training complete')
print(f'{len(miner.drain.clusters)} template unik | state → {DRAIN_STATE.name}')

## 4. Tahap 3 & 4 — Sliding window + split

In [ ]:
import random
df = df.sort_values('timestamp_unix').reset_index(drop=True)
df['win_idx'] = df['timestamp_unix'] // (WINDOW_MINUTES * 60)
windows = [{
    'log_keys': g['log_key'].astype(str).tolist(),
    'is_anomaly': bool(g['is_anomaly'].any()),
} for _, g in df.groupby('win_idx', sort=True)]

n_anom = sum(w['is_anomaly'] for w in windows)
print(f'{len(windows):,} window | anomali {n_anom:,} ({n_anom/len(windows):.2%})')

random.seed(SEED)
normal = [w for w in windows if not w['is_anomaly']]; random.shuffle(normal)
train_data = normal[:int(len(normal)*TRAIN_RATIO)]
if SMOKE: train_data = train_data[:200]
print(f'Train (normal): {len(train_data):,}' + (' [SMOKE]' if SMOKE else ''))

## 5. Tahap 5 — Training LogBERT (MLKP, loop manual)

In [ ]:
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForMaskedLM, DataCollatorForLanguageModeling

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
tok = AutoTokenizer.from_pretrained(BASE_MODEL)

# MLKP: tiap Log Key unik = 1 token
unique_keys = sorted({k for w in train_data for k in w['log_keys']})
tok.add_tokens([f'{LOGKEY_PREFIX}{k}' for k in unique_keys])
print(f'{len(unique_keys)} Log Key unik ditambahkan')

def seq2txt(w):
    return ' '.join(f'{LOGKEY_PREFIX}{k}' for k in w['log_keys'][:MAX_LOG_KEYS])

class DS(torch.utils.data.Dataset):
    def __init__(s, ws):
        s.enc = tok([seq2txt(w) for w in ws], truncation=True,
                    padding='max_length', max_length=MAX_LENGTH)
    def __len__(s): return len(s.enc['input_ids'])
    def __getitem__(s, i): return {k: torch.tensor(v[i]) for k, v in s.enc.items()}

coll = DataCollatorForLanguageModeling(tok, mlm=True, mlm_probability=MLM_PROBABILITY)
loader = DataLoader(DS(train_data), batch_size=BATCH_SIZE, shuffle=True, collate_fn=coll)

model = AutoModelForMaskedLM.from_pretrained(BASE_MODEL)
model.resize_token_embeddings(len(tok)); model.to(dev).train()
opt = torch.optim.AdamW(model.parameters(), lr=5e-5)
scaler = torch.amp.GradScaler('cuda', enabled=dev=='cuda')
epochs = 1 if SMOKE else EPOCHS

for ep in range(epochs):
    run, n = 0.0, 0; opt.zero_grad()
    for step, b in enumerate(loader):
        b = {k: v.to(dev) for k, v in b.items()}
        with torch.amp.autocast('cuda', enabled=dev=='cuda'):
            loss = model(**b).loss / GRAD_ACCUM
        scaler.scale(loss).backward()
        if (step+1) % GRAD_ACCUM == 0:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); opt.zero_grad()
        run += loss.item()*GRAD_ACCUM; n += 1
        if step % 200 == 0: print(f'  ep {ep+1} step {step}/{len(loader)} loss {loss.item()*GRAD_ACCUM:.3f}')
    print(f'  ep {ep+1} avg loss {run/max(n,1):.4f}')

model.save_pretrained(LOGBERT_DIR); tok.save_pretrained(LOGBERT_DIR)
print('LogBERT →', LOGBERT_DIR)

## 6. Tahap 6 — DeepSVDD (PyTorch murni)
Definisi jaringan **identik** dengan `src/model/svdd.py` lokal agar `svdd.pt` bisa dimuat `predict.py`.

In [ ]:
import numpy as np, torch.nn as nn
from transformers import AutoModel

class DeepSVDDNet(nn.Module):
    def __init__(self, in_dim, rep_dim=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, 128, bias=False), nn.ReLU(),
                                 nn.Linear(128, rep_dim, bias=False))
    def forward(self, x): return self.net(x)

bert = AutoModel.from_pretrained(LOGBERT_DIR).eval().to(dev)   # BertModel dari config

def embed(ws, batch=64):
    out = []
    for i in range(0, len(ws), batch):
        enc = tok([seq2txt(w) for w in ws[i:i+batch]], return_tensors='pt',
                  truncation=True, padding=True, max_length=MAX_LENGTH).to(dev)
        with torch.no_grad():
            out.append(bert(**enc).last_hidden_state[:, 0, :].cpu().numpy())
    return np.vstack(out)

X = embed(train_data); print('Embedding:', X.shape)

Xt = torch.as_tensor(X, dtype=torch.float32, device=dev)
net = DeepSVDDNet(Xt.shape[1]).to(dev)
with torch.no_grad():
    c = net(Xt).mean(0); c[c.abs() < 1e-6] = 1e-6
opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-5); net.train()
for _ in range(1 if SMOKE else SVDD_EPOCHS):
    opt.zero_grad(); loss = ((net(Xt)-c)**2).sum(1).mean(); loss.backward(); opt.step()
net.eval()
with torch.no_grad(): scores = ((net(Xt)-c)**2).sum(1).cpu().numpy()

torch.save({'state_dict': net.state_dict(), 'center': c.detach().cpu(),
            'in_dim': X.shape[1], 'score_min': float(scores.min()),
            'score_max': float(scores.max())}, SVDD_MODEL)
print(f'DeepSVDD → {SVDD_MODEL} | rentang [{scores.min():.4f}, {scores.max():.4f}]')

## 7. Bungkus & unduh artefak
Simpan ke Drive (aman dari disconnect) **dan** sediakan `models.zip` untuk diunduh.

In [ ]:
import shutil
shutil.make_archive('/content/models', 'zip', '/content/models')

# Salinan cadangan ke Drive
shutil.copy('/content/models.zip', '/content/drive/MyDrive/models.zip')
print('Tersimpan: /content/drive/MyDrive/models.zip')

# Unduh langsung ke komputer
from google.colab import files
files.download('/content/models.zip')

## 8. Di komputer lokal

1. Ekstrak `models.zip` ke folder project → `DeepL_FP/models/`
   (isi: `logbert/`, `svdd.pt`, `drain3_state.bin`).
2. Uji: `python -c "from src.model.predict import predict; print(predict(['RAS KERNEL FATAL uncorrectable memory error']))"`
3. Ganti stub di pemanggil: `from src.model.predict import predict`.

`predict.py` tidak meng-import `datasets`/`pyod`/`Trainer`, jadi inference lokal aman
meski env Windows-mu bermasalah untuk training.